In [1]:
import torch as th
from torch import nn

from music_diffusion.networks.unet import TimeUNet
from music_diffusion.networks.liquid import LiquidRecurrent

In [2]:
channels = 2
nb_stft_vec = 1024
nfft = 512

steps = 4096

In [3]:
unet = TimeUNet(
    [
            (2, 16),
            (16, 32),
            (32, 64),
            (64, 128),
            (128, 256),
            (256, 512),
        ],
    [2, 4, 8, 16, 32, 64],
    128,
    steps,
)

In [4]:
batch_size = 3
step_batch_size = 1

x_t = th.randn(batch_size, step_batch_size, channels, nfft, nb_stft_vec)
t = th.randint(0, steps, (batch_size, step_batch_size))

In [5]:
tmp = unet(x_t, t)

In [6]:
tmp.size()

torch.Size([3, 1, 512, 8, 16])

In [22]:
def process_time(x: th.Tensor, ltc: LiquidRecurrent) -> th.Tensor:
    curr_batch_size, curr_step_batch_size = x.size()[:2]
    prepared_x = th.flatten(x, 0, 1)
    prepared_x = th.permute(prepared_x, (0, 2, 3, 1))

    mixed_batch_size, compressed_nfft = prepared_x.size()[:2]
    prepared_x = th.flatten(prepared_x, 0, 1)

    out = ltc(prepared_x)

    out = th.unflatten(out, 0, (mixed_batch_size, compressed_nfft))
    out = th.permute(out, (0, 3, 1, 2))
    out = th.unflatten(out, 0, (curr_batch_size, curr_step_batch_size))

    return out

In [23]:
neuron_number = 32

time_ltc = LiquidRecurrent(neuron_number, 512, 512, 6, 1.0)

In [24]:
time_output = process_time(tmp, time_ltc)

In [25]:
time_output.size()

torch.Size([3, 1, 512, 8, 16])

In [51]:
def process_stft(x: th.Tensor, ltc: LiquidRecurrent) -> th.Tensor:
    curr_batch_size, curr_step_batch_size = x.size()[:2]
    prepared_x = th.flatten(x, 0, 1)
    prepared_x = th.permute(prepared_x, (0, 3, 2, 1))

    mixed_batch_size, compressed_time_space = prepared_x.size()[:2]
    prepared_x = th.flatten(prepared_x, 0, 1)

    out = ltc(prepared_x)

    out = th.unflatten(out, 0, (mixed_batch_size, compressed_time_space))
    out = th.permute(out, (0, 3, 2, 1))
    out = th.unflatten(out, 0, (curr_batch_size, curr_step_batch_size))

    return out

In [52]:
stft_ltc = LiquidRecurrent(neuron_number, 512, 512, 6, 1.0)

In [53]:
stft_output = process_stft(time_output, stft_ltc)

In [54]:
print(stft_output.size())

torch.Size([3, 1, 512, 8, 16])
